In [1]:
import pandas as pd
import numpy as np

In [2]:
primekg = pd.read_csv('dataverse_files/kg.csv')

host_layer = pd.read_csv('t1dm_host_layer_data.csv')
drug_data = pd.read_csv('drug_data_14_07.csv')

antibiotics_df = pd.read_csv('antibiotics_list.csv')

/tmp/ipykernel_325860/1665257926.py:1: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  primekg = pd.read_csv('dataverse_files/kg.csv')


In [3]:
gene_listx = host_layer[host_layer['x_type'] == 'gene/protein']['x_name'].to_list()
gene_listy = host_layer[host_layer['y_type'] == 'gene/protein']['y_name'].to_list()

In [4]:
gene_list = list(set(gene_listx + gene_listy))

In [5]:
relevant_drugs = primekg[(primekg['x_type'] == 'drug') & (primekg['y_name'].isin(gene_list))].copy()

In [6]:
relevant_drugs['drug_class'] = np.zeros(len(relevant_drugs))

In [7]:
relevant_drugs[relevant_drugs['x_name'] == 'Ciprofloxacin']

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class
323522,drug_protein,enzyme,14188,DB00537,drug,Ciprofloxacin,DrugBank,3955,1544,gene/protein,CYP1A2,NCBI,0.0
333070,drug_protein,target,14188,DB00537,drug,Ciprofloxacin,DrugBank,6976,7153,gene/protein,TOP2A,NCBI,0.0


In [8]:
antibiotics_df.tail(3)

,Name,Family,Usage,Popular Brand Name
102,Tedizolid,Oxazolidinone,Skin infections,Sivextro
103,Ceftaroline fosamil,Cephalosporin,"Skin infections, community-acquired pneumonia",Teflaro
104,Solithromycin,Macrolide,Community-acquired bacterial pneumonia,Solithera


In [9]:
antibiotic_name_list = antibiotics_df['Name'].to_list()

In [10]:
def drug_class_fxn(row): 
    res = row.str.contains('|'.join(antibiotic_name_list))
    if res['x_name'] == True:
        for_compoundnames = row['x_name'].split(' ')
        index = antibiotic_name_list.index(for_compoundnames[0])
        row['drug_class'] = antibiotics_df.iloc[index,1]

    return row

In [11]:
filled_drug_class = relevant_drugs.apply(lambda x: drug_class_fxn(x), axis = 1)

In [12]:
#keeping only those we could succesfully match
filled_drug_class = filled_drug_class[filled_drug_class['drug_class'] != 0]

In [13]:
filled_drug_class.tail()

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class
343499,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,763,5601,gene/protein,MAPK9,NCBI,Tetracycline
345908,drug_protein,transporter,14309,DB01137,drug,Levofloxacin,DrugBank,5481,6582,gene/protein,SLC22A2,NCBI,Fluoroquinolone
345932,drug_protein,transporter,15044,DB11633,drug,Isavuconazole,DrugBank,5481,6582,gene/protein,SLC22A2,NCBI,Azole
345971,drug_protein,transporter,15443,DB00520,drug,Caspofungin,DrugBank,12713,6580,gene/protein,SLC22A1,NCBI,Echinocandin
345996,drug_protein,transporter,14309,DB01137,drug,Levofloxacin,DrugBank,12713,6580,gene/protein,SLC22A1,NCBI,Fluoroquinolone


In [14]:
key_classes = drug_data['Drug_Class'].to_list()

In [15]:
filled_drug_class_key = filled_drug_class[filled_drug_class['drug_class'].str.contains('|'.join(key_classes), case = False)].copy()

In [16]:
filled_drug_class_key['drug_id'] = np.zeros(len(filled_drug_class_key))

In [17]:
filled_drug_class_key['drug_type'] = ['drug_class']*len(filled_drug_class_key)

In [18]:
filled_drug_class_key.head()

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class,drug_id,drug_type
322065,drug_protein,enzyme,14597,DB00250,drug,Dapsone,DrugBank,4425,5743,gene/protein,PTGS2,NCBI,Sulfone,0.0,drug_class
322387,drug_protein,enzyme,14597,DB00250,drug,Dapsone,DrugBank,4818,4353,gene/protein,MPO,NCBI,Sulfone,0.0,drug_class
322745,drug_protein,enzyme,14318,DB01212,drug,Ceftriaxone,DrugBank,6418,2752,gene/protein,GLUL,NCBI,Cephalosporin,0.0,drug_class
323468,drug_protein,enzyme,15289,DB00218,drug,Moxifloxacin,DrugBank,3955,1544,gene/protein,CYP1A2,NCBI,Fluoroquinolone,0.0,drug_class
323503,drug_protein,enzyme,15299,DB00440,drug,Trimethoprim,DrugBank,3955,1544,gene/protein,CYP1A2,NCBI,Sulfonamide,0.0,drug_class


In [19]:
#using drug_dict from the AMR cleaning notebook for ease of ID assignment
drug_dict = {'carbapenem':'160000', 'diaminopyrimidine':'160001', 'cephalosporin':'160002',
       'tetracycline':'160003', 'fusidane':'160004',
       'lincosamide':'160005', 'sulfonamide':'160006',
       'macrolide':'160007', 'phosphonic acid':'160008',
       'disinfecting agents and antiseptics':'160009',
       'aminoglycoside':'160010', 'glycopeptide':'160011',
       'peptide':'160012', 'fluoroquinolone':'160013',
       'isoniazid-like':'160014', 'penam':'160015', 'rifamycin':'160016',
       'phenicol':'160017', 'glycylcycline':'160018', 'polyamine':'160019',
       'cephamycin':'160020', 'streptogramin A':'160021',
       'aminocoumarin':'160022', 'pyrazine':'160023',
       'nucleoside':'160024', 'streptogramin':'160025',
       'pleuromutilin':'160026', 'elfamycin':'160027',
       'oxazolidinone':'160028', 'nitrofuran':'160029',
       'mupirocin-like':'160030', 'bicyclomycin-like':'160031',
       'nitroimidazole':'160032', 'salicylic acid':'160033',
       'monobactam':'160034', 'penem':'160035', 'sulfone':'160036',
       'thioamide':'160037', 'streptogramin B':'160038',
       'nybomycin-like':'160039', 'oxacephem':'160040'}

In [20]:
filled_drug_class_key['drug_class'] = filled_drug_class_key['drug_class'].str.lower()

In [21]:
filled_drug_class_key

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class,drug_id,drug_type
322065,drug_protein,enzyme,14597,DB00250,drug,Dapsone,DrugBank,4425,5743,gene/protein,PTGS2,NCBI,sulfone,0.0,drug_class
322387,drug_protein,enzyme,14597,DB00250,drug,Dapsone,DrugBank,4818,4353,gene/protein,MPO,NCBI,sulfone,0.0,drug_class
322745,drug_protein,enzyme,14318,DB01212,drug,Ceftriaxone,DrugBank,6418,2752,gene/protein,GLUL,NCBI,cephalosporin,0.0,drug_class
323468,drug_protein,enzyme,15289,DB00218,drug,Moxifloxacin,DrugBank,3955,1544,gene/protein,CYP1A2,NCBI,fluoroquinolone,0.0,drug_class
323503,drug_protein,enzyme,15299,DB00440,drug,Trimethoprim,DrugBank,3955,1544,gene/protein,CYP1A2,NCBI,sulfonamide,0.0,drug_class
323522,drug_protein,enzyme,14188,DB00537,drug,Ciprofloxacin,DrugBank,3955,1544,gene/protein,CYP1A2,NCBI,fluoroquinolone,0.0,drug_class
323603,drug_protein,enzyme,14309,DB01137,drug,Levofloxacin,DrugBank,3955,1544,gene/protein,CYP1A2,NCBI,fluoroquinolone,0.0,drug_class
325521,drug_protein,enzyme,14597,DB00250,drug,Dapsone,DrugBank,8212,1571,gene/protein,CYP2E1,NCBI,sulfone,0.0,drug_class
327266,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,373,4843,gene/protein,NOS2,NCBI,tetracycline,0.0,drug_class
327847,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,3474,4318,gene/protein,MMP9,NCBI,tetracycline,0.0,drug_class


In [22]:
filled_drug_class_key['drug_class'] = filled_drug_class_key['drug_class'].replace({'lipoglycopeptide':'glycopeptide', 'amphenicol':'phenicol', 'polypeptide':'peptide'})

#justification
#https://www.msdmanuals.com/home/infections/antibiotics/polypeptides?_gl=1*bqonyv*_up*MQ..*_ga*MTQ5MTM0Mjk0My4xNzg0MTExNDk1*_ga_CJ792HFYYC*czE3ODQxMTE0OTQkbzEkZzAkdDE3ODQxMTE0OTQkajYwJGwwJGgw
#https://bio.libretexts.org/Bookshelves/Microbiology/Microbiology_(Kaiser)/Unit_7%3A_Microbial_Genetics_and_Microbial_Metabolism/19%3A_Review_of_Molecular_Genetics/19.1%3A_Polypeptides_and_Proteins
#https://www.msdvetmanual.com/pharmacology/antibacterial-agents/phenicols-use-in-animals
#https://en.wikipedia.org/wiki/Amphenicol

In [23]:
#Filling the main dataframe with created IDs
def id_fxn(row):
    row['drug_id'] = drug_dict[row['drug_class']]

    return row

filled_drug_class_key = filled_drug_class_key.apply(lambda x: id_fxn(x), axis = 1)

In [24]:
#checker
filled_drug_class_key[filled_drug_class_key['drug_id'] == 0]

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class,drug_id,drug_type


In [25]:
filled_drug_class_key = filled_drug_class_key.groupby(filled_drug_class_key[['drug_class','y_name']].agg(frozenset,axis = 1)).first().reset_index(drop=True)

In [26]:
filled_drug_class_key

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class,drug_id,drug_type
0,drug_protein,enzyme,14597,DB00250,drug,Dapsone,DrugBank,4425,5743,gene/protein,PTGS2,NCBI,sulfone,160036,drug_class
1,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,763,5601,gene/protein,MAPK9,NCBI,tetracycline,160003,drug_class
2,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,902,5599,gene/protein,MAPK8,NCBI,tetracycline,160003,drug_class
3,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,2275,5598,gene/protein,MAPK7,NCBI,tetracycline,160003,drug_class
4,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,1447,5597,gene/protein,MAPK6,NCBI,tetracycline,160003,drug_class
5,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,866,5595,gene/protein,MAPK3,NCBI,tetracycline,160003,drug_class
6,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,197,1432,gene/protein,MAPK14,NCBI,tetracycline,160003,drug_class
7,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,7895,5603,gene/protein,MAPK13,NCBI,tetracycline,160003,drug_class
8,drug_protein,target,15914,DB01017,drug,Minocycline,DrugBank,418,5594,gene/protein,MAPK1,NCBI,tetracycline,160003,drug_class
9,drug_protein,target,18036,DB00626,drug,Bacitracin,DrugBank,4596,3416,gene/protein,IDE,NCBI,peptide,160012,drug_class


In [27]:
filled_drug_class_key = filled_drug_class_key[['relation','display_relation','drug_id','drug_type','drug_class','y_index','y_type','y_name']]

In [28]:
final_drugclass_gene_data = filled_drug_class_key.rename(columns={'drug_id':'x_index','drug_type':'x_type','drug_class':'x_name'})

In [29]:
final_drugclass_gene_data.to_csv('t1dm_drugclass_gene_data.csv', index = False)